In [8]:
import numpy as np
import pandas as pd
import random
import time

# Start the timer
start_time = time.perf_counter()

# 1. Initialize a list to hold results from each repetition
results = []

n_repetitions = 1000

for i in range(n_repetitions):
    # data_seed = 456 # 可以用同一個邏輯獨立產生不同分布
    def generate_data(seed):
        num_subjects = 1000
        num_features = 10

        rng = np.random.default_rng(seed)

        X_raw = rng.standard_normal(
            (num_subjects, num_features)
        )

        ones_column = np.ones(
            (num_subjects, 1)
        )

        X = np.concatenate(
            (ones_column, X_raw),
            axis=1
        )
    
        # 第一個是 intercept
        beta_true = np.array([
            1.0,   # intercept
            2.0,   # X1 effect
            -1.5,  # X2 effect
            0.5,   # X3 effect
            0.0,   # X4 no effect
            3.0,   # X5 effect
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
        ]).reshape(-1, 1)
    
    
        noise = rng.standard_normal(
            (num_subjects, 1)
        )
    
        Y = (
            X @ beta_true
            + noise
            + X[:, 2:3] * X[:, 3:4] # X2 * X3 interaction
            + X[:, 4:5] * X[:, 4:5] # X4^2 quadratic term
        )
    
        columns = ["Intercept"] + [f"X{i}" for i in range(1, num_features+1)]
    
        data = pd.DataFrame(X, columns=columns)
    
        data["Y"] = Y.flatten()
    
        return data, beta_true
    
    # 先讀進模擬資料
    
    #data, beta_true = generate_data(seed = data_seed)
    
    data, beta_true = generate_data(random.seed(None))
    
    
    cols = data.columns[:-1].to_list() 
    
    whole = data.copy()
    
    data = whole.iloc[0:700,:]
    
    validation = whole.iloc[700:1000,:]
    
    import pandas as pd
    import numpy as np
    
    # ### 先用傳統統計模型驗證
    X = data[cols]
    
    XTX = X.T @ X
    
    # 用套件驗證
    import statsmodels.api as sm
    
    X = data[cols]
    
    model = sm.OLS(
        data["Y"],
        X          # 不加 constant
    )
    
    result = model.fit()
    
    from sklearn.linear_model import Ridge
    from sklearn.metrics import mean_squared_error, r2_score
    
    
    # ======================
    # Split X and Y
    # ======================
    
    X_train = data[cols].values
    y_train = data["Y"].values
    
    # X_test = validation[cols].values
    # y_test = validation["Y"].values
    
    
    # ======================
    # Ridge model
    # ======================
    
    ridge = Ridge(alpha=1.0, fit_intercept=False)
    
    ridge.fit(
        X_train,
        y_train,
    )
    
    import torch
    
    X = torch.tensor(
        data[cols].values,
        dtype=torch.float32
    )
    
    y = torch.tensor(
        data["Y"].values,
        dtype=torch.float32
    )
    
    N = X.shape[0]
    P = X.shape[1]
    
    import torch.nn as nn
    
    y = y.reshape(-1, 1)
    
    X_Y_train = torch.cat(
        (X, y),
        dim=1
    )
    
    X_Y_features = X_Y_train.t()
    
    d_k = 49
    
    # d_k = 32**2
    W_Q = nn.Linear(
        X_Y_features.shape[1],
        d_k,
        bias=False
    )
    
    W_K = nn.Linear(
        X_Y_features.shape[1],
        d_k,
        bias=False
    )
    
    optimizer = torch.optim.Adam(
    
        list(W_Q.parameters()) +
        list(W_K.parameters()), 
        
        lr=0.001
    
    )
    
    import numpy as np
    import torch.nn.functional as F
    
    lam = 1
    
    # Deep GLM 
    p = X.shape[1]
    
    I = torch.eye(
        p,
        dtype=X.dtype,
        device=X.device
    )
    
    loss_history = []
    
    initial_attn = None
    
    epochs = 1000
    
    for epoch in range(epochs):
    
        Q = W_Q(X_Y_features)
        
        K = W_K(X_Y_features)
    
        scores = torch.matmul(
            Q,
            K.transpose(-2,-1)
        ) 
    
        # scores = scores / np.sqrt(dk)
    
        attention_matrix = F.softmax(
            scores / (d_k** 0.5 ), # d_k ** 0.5，不做開根號
            dim=-1
        )
    
        # 把Y再從矩陣中拿掉
        A = attention_matrix[:-1,:-1]
    
        # 矩陣乘上Y的變異數
        var_y = torch.var(y)
    
        A_var_y = A * var_y
    
    
        # beta
    
        beta = torch.linalg.solve(X.T @ X + (I + A).T@(I+A),X.T @ y)

        y_hat = X @ beta
    
        loss = F.mse_loss(
            y_hat,
            y
        )
    
    
        if epoch == 0:
            initial_attn = attention_matrix.detach().clone()
    
    
        loss_history.append(
            loss.item()
        )
    
        optimizer.zero_grad()
    
        loss.backward()
    
        optimizer.step()
    
    # =====================
    # 訓練完成後重新 forward
    # 取得最後 attention
    # =====================
    
    with torch.no_grad():
    
        Q = W_Q(X_Y_features)
        
        K = W_K(X_Y_features)
    
        scores = torch.matmul(
            Q,
            K.transpose(-2,-1)
        ) 
    
        # scores = scores / np.sqrt(dk)
    
        attention_matrix = F.softmax(
            scores / (d_k** 0.5 ), # d_k ** 0.5，不做開根號
            dim=-1
        )
    
        # 把Y再從矩陣中拿掉
        A = attention_matrix[:-1,:-1]
    
        final_attn = A.clone()
    
        # 矩陣乘上Y的變異數
        var_y = torch.var(y)
    
        # A_var_y = A * var_y
    
    
        # beta
    
        beta1 = torch.linalg.solve(X.T @ X + (I + A).T@(I+A),X.T @ y)

        # adaptive
    
        adaptive = I + A.T@A/torch.trace(A)
    
    
    attn_matrix = final_attn.detach().flatten()
    
    ### 做驗證
    
    beta_ols = result.params.values
    beta_attn = beta1.detach().numpy()
    beta_attn = beta_attn.flatten()
    beta_ridge = ridge.coef_
    
    X_val = validation[cols].values
    y_val = validation["Y"].values
    
    y_pred_ols = X_val @ beta_ols
    
    y_pred_attn = X_val @ beta_attn
    
    y_pred_ridge = X_val @ beta_ridge
    
    from sklearn.metrics import mean_squared_error
    
    mse_ols = mean_squared_error(
        y_val,
        y_pred_ols
    )
    
    mse_attn = mean_squared_error(
        y_val,
        y_pred_attn
    )
    
    mse_ridge = mean_squared_error(
        y_val,
        y_pred_ridge
    )
    
    rmse_ols = np.sqrt(mse_ols)
    rmse_attn = np.sqrt(mse_attn)
    rmse_ridge = np.sqrt(mse_ridge)
    
    from sklearn.metrics import r2_score
    
    ols_r2 = r2_score(y_val, y_pred_ols)
    attn_r2 = r2_score(y_val, y_pred_attn)
    ridge_r2 = r2_score(y_val, y_pred_ridge)
    
    # 模擬資料生成時的實際係數
    beta_true = beta_true.flatten()
    
    ols_bias = abs(beta_ols - beta_true)
    attn_bias = abs(beta_attn - beta_true)
    ridge_bias = abs(beta_ridge - beta_true)
    
    beta_table = pd.DataFrame({
        "Variable": cols,
        "OLS_beta": beta_ols,
        "Ridge_beta" : beta_ridge,
        "Attention_beta": beta_attn,
        "Simulation_beta": beta_true
    })

    # --- YOUR MODEL FITTING & METRIC COMPUTATION HERE ---
    # (Assuming mse_ols, rmse_ols, ols_r2, etc. are calculated in each loop)

    # 2. Append metrics as a dictionary for this iteration
    results.append(
        {
            "OLS_MSE": mse_ols,
            "OLS_RMSE": rmse_ols,
            "OLS_R2": ols_r2,
            "Ridge_MSE": mse_ridge,
            "Ridge_RMSE": rmse_ridge,
            "Ridge_R2": ridge_r2,
            "Attention_MSE": mse_attn,
            "Attention_RMSE": rmse_attn,
            "Attention_R2": attn_r2,
        }
    )

# 3. Create a DataFrame containing all 1000 repetitions
df_results = pd.DataFrame(results)

# 4. Calculate average metrics across all 1000 runs
avg_metrics = df_results.mean()
var_metrics = df_results.var()

# 5. Reshape into a tidy summary table
# Helper function to create model-level DataFrames
def build_summary_table(stats_series):
    return pd.DataFrame(
        {
            "Model": ["OLS", "Ridge", "Attention"],
            "MSE": [
                stats_series["OLS_MSE"],
                stats_series["Ridge_MSE"],
                stats_series["Attention_MSE"],
            ],
            "RMSE": [
                stats_series["OLS_RMSE"],
                stats_series["Ridge_RMSE"],
                stats_series["Attention_RMSE"],
            ],
            "R2": [
                stats_series["OLS_R2"],
                stats_series["Ridge_R2"],
                stats_series["Attention_R2"],
            ],
        }
    )

mean_df = build_summary_table(avg_metrics)
var_df = build_summary_table(var_metrics)


# 6. Display the final average results table
print(f"=== Mean Metrics ({n_repetitions} Repetitions) ===")
print(mean_df.to_string(index=False, float_format=lambda x: f"{x:12.6f}"))

print(f"\n=== Variance Metrics ({n_repetitions} Repetitions) ===")
print(var_df.to_string(index=False, float_format=lambda x: f"{x:12.6e}"))


# End the timer
end_time = time.perf_counter()

# Calculate the difference
execution_time = end_time - start_time
print(f"Execution time: {execution_time:.6f} seconds")





=== Mean Metrics (1000 Repetitions) ===
    Model          MSE         RMSE           R2
      OLS     4.104649     2.021723     0.788277
    Ridge     4.104548     2.021694     0.788286
Attention     4.104493     2.021682     0.788287

=== Variance Metrics (1000 Repetitions) ===
    Model          MSE         RMSE           R2
      OLS 2.915376e-01 1.730314e-02 7.860923e-04
    Ridge 2.918000e-01 1.731892e-02 7.852491e-04
Attention 2.917003e-01 1.731318e-02 7.856495e-04
Execution time: 3436.023998 seconds
